## CUDA Stream 基础概念
###  什么是 CUDA Stream？
- CUDA Stream 是 GPU 上按顺序执行的一系列操作队列。<font color='red'>同一 Stream 内的操作保证按提交顺序执行，不同 Stream 之间的操作可以并发执行（在资源允许的情况下）</font>。

In [ ]:
Stream A: [Kernel1] → [Kernel2] → [Kernel3]  (顺序执行)
Stream B: [Kernel4] → [Kernel5]              (顺序执行)
           ↑ 与 Stream A 并发执行

### 流的三种类型

| 类型                            | 说明                                      |
| ----------------------------- | --------------------------------------- |
| **Default Stream (Stream 0)** | 每个设备有一个，同步流。传统上阻塞其他流，但 CUDA 7+ 支持非阻塞默认流 |
| **Blocking Stream**           | 会阻塞默认流，也会被默认流阻塞（旧行为）                    |
| **Non-blocking Stream**       | 与默认流互不阻塞，真正的并发                          |


## torch.cuda.default_stream() 详解
### 函数签名

In [ ]:
torch.cuda.default_stream(device: Optional[Union[int, torch.device]] = None) -> torch.cuda.Stream

- device：指定设备索引或 torch.device 对象。默认为 None，即当前设备。
- 返回值：torch.cuda.Stream 对象，表示该设备的默认流。

### 核心特性

In [ ]:
import torch

# 获取当前设备（cuda:0）的默认流
default_stream = torch.cuda.default_stream()
print(default_stream)  
# 输出: <torch.cuda.Stream object at 0x...> (cuda:0, default_stream)

# 获取指定设备的默认流
stream_cuda1 = torch.cuda.default_stream(1)  # cuda:1 的默认流

# 验证默认流的属性
print(default_stream.device)       # cuda:0
print(default_stream.cuda_stream)  # 底层 CUDA stream 句柄 (c_void_p)
print(default_stream.priority)     # 优先级 (通常为 0)

### 默认流 vs 普通流

In [ ]:
import torch

# 默认流（每个设备一个，始终存在）
default_s = torch.cuda.default_stream()

# 创建新的普通流
user_stream = torch.cuda.Stream()

print(default_s == user_stream)  # False

# 关键区别：默认流是"隐式"的
# 所有未指定 stream 的 CUDA 操作都在默认流上执行
x = torch.randn(1000, 1000, device='cuda')  # 在默认流上
y = x @ x.T                                   # 在默认流上

## 完整使用示例
### 基本用法：确认操作所在的流

In [ ]:
import torch

def check_stream(tensor, label):
    """检查张量最后一次操作所在的流"""
    print(f"{label}: stream = {tensor.grad_fn.metadata.get('stream', 'unknown') if tensor.grad_fn else 'leaf'}")

# 获取默认流
default_stream = torch.cuda.default_stream()

print(f"Default stream: {default_stream}")

# 所有操作默认在 default_stream 上
a = torch.randn(1000, 1000, device='cuda')
b = torch.randn(1000, 1000, device='cuda')

c = a + b        # 在 default_stream 上
d = c @ b        # 在 default_stream 上

print(f"Stream of d: {d.device}")  # cuda:0

### 显式使用默认流进行同步

In [ ]:
import torch
import time

# 创建一个新流
stream1 = torch.cuda.Stream()

# 在 stream1 上执行异步操作
with torch.cuda.stream(stream1):
    x = torch.randn(5000, 5000, device='cuda')
    y = x @ x.T  # 大矩阵乘法，耗时较长

# 此时 y 的计算可能还没完成（异步）
print("计算已提交，但可能尚未完成")

# 方式1：等待 stream1 完成
stream1.synchronize()
print("stream1 已完成")

# 方式2：让默认流等待 stream1（确保默认流上的后续操作在 stream1 完成后执行）
default_stream = torch.cuda.default_stream()
default_stream.wait_stream(stream1)

# 现在在默认流上使用 y 是安全的
z = y + 1  # 在默认流上，但会等 stream1 完成

### 多流并发 vs 默认流的阻塞行为

In [ ]:
import torch
import time

def benchmark_stream(stream_type):
    """对比默认流和用户流的性能"""
    size = 5000
    
    a = torch.randn(size, size, device='cuda')
    b = torch.randn(size, size, device='cuda')
    
    # 预热 全局同步	等待所有流完成
    torch.cuda.synchronize()
    
    if stream_type == 'default':
        # 顺序在默认流上执行
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        
        start.record()
        c = a @ b  # op1
        d = c @ b  # op2 (必须等 op1)
        e = d @ b  # op3 (必须等 op2)
        end.record()
        
    elif stream_type == 'concurrent':
        # 创建两个流，并发执行
        s1 = torch.cuda.Stream()
        s2 = torch.cuda.Stream()
        
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        
        start.record()
        
        # 拆分计算到两个流
        with torch.cuda.stream(s1):
            c1 = a[:size//2, :] @ b  # 上半部分
        
        with torch.cuda.stream(s2):
            c2 = a[size//2:, :] @ b  # 下半部分
        
        # 默认流等待两个流
        default = torch.cuda.default_stream()
        default.wait_stream(s1)
        default.wait_stream(s2)
        
        # 合并结果（在默认流上）
        c = torch.cat([c1, c2], dim=0)
        
        end.record()
    
    torch.cuda.synchronize()
    return start.elapsed_time(end)

# 运行对比
t_default = benchmark_stream('default')
t_concurrent = benchmark_stream('concurrent')

print(f"Default stream sequential: {t_default:.2f} ms")
print(f"Concurrent streams: {t_concurrent:.2f} ms")
print(f"Speedup: {t_default/t_concurrent:.2f}x")

###  默认流在 Megatron-LM 中的实际应用

In [ ]:
# Megatron-LM 中典型的流使用模式
from megatron.core import parallel_state
import torch

def forward_backward_pipeline_stage(
    hidden_states,
    model_chunk,
    forward_only=False
):
    """
    Megatron 流水线并行中的流管理
    """
    # 获取当前设备的默认流
    default_stream = torch.cuda.default_stream()
    
    # 前向传播在默认流上
    with torch.cuda.stream(default_stream):
        output = model_chunk(hidden_states)
    
    if not forward_only:
        # 反向传播可能需要单独的流来重叠通信和计算
        # 但梯度计算本身仍在默认流上（或当前活跃流）
        output.backward(grad_output)
    
    return output

# 更复杂的：计算与通信重叠
def overlap_communication_and_compute(tensor, comm_group):
    """
    使用多流重叠 AllReduce 通信和计算
    """
    default_stream = torch.cuda.default_stream()
    comm_stream = torch.cuda.Stream()
    
    # 在默认流上继续计算
    with torch.cuda.stream(default_stream):
        computed = tensor * 2  # 一些计算
    
    # 在通信流上发起 AllReduce
    with torch.cuda.stream(comm_stream):
        handle = torch.distributed.all_reduce(tensor, group=comm_group, async_op=True)
    
    # 默认流等待通信完成（如果需要 tensor 的结果）
    default_stream.wait_stream(comm_stream)
    
    return computed

## 默认流的关键行为
### 隐式性

In [ ]:
import torch

# 以下三种写法等价：都在默认流上执行
x1 = torch.randn(100, device='cuda')

torch.cuda.set_device(0)
x2 = torch.randn(100, device='cuda')

default_s = torch.cuda.default_stream()
with torch.cuda.stream(default_s):
    x3 = torch.randn(100, device='cuda')

### 同步语义

In [ ]:
import torch

s1 = torch.cuda.Stream()

# 旧 CUDA 行为（< 7.0）：默认流会阻塞所有其他流
# 新 CUDA 行为（>= 7.0）：默认流也是普通流，不特殊

# 显式同步：让默认流等待 s1
default = torch.cuda.default_stream()
default.wait_stream(s1)

# 显式同步：让 s1 等待默认流
s1.wait_stream(default)

# 全局同步：阻塞 CPU 直到默认流完成
torch.cuda.synchronize()  # 等价于 default_stream.synchronize()

# 仅同步默认流
default.synchronize()

### 设备绑定

In [ ]:
import torch

# 每个设备有独立的默认流
s0 = torch.cuda.default_stream(0)  # cuda:0 的默认流
s1 = torch.cuda.default_stream(1)  # cuda:1 的默认流

print(s0.device)  # cuda:0
print(s1.device)  # cuda:1

# 切换当前设备后获取
torch.cuda.set_device(1)
s1_alt = torch.cuda.default_stream()  # 也是 cuda:1 的默认流
print(s1 == s1_alt)  # True

## 与 torch.cuda.current_stream() 的区别

| 函数                 | 返回               | 是否可变                    |
| ------------------ | ---------------- | ----------------------- |
| `default_stream()` | 设备的默认流（Stream 0） | ❌ 固定                    |
| `current_stream()` | 当前线程的活跃流         | ✅ 可通过 `set_stream()` 改变 |


In [ ]:
import torch

# default_stream(): 获取设备的默认流（始终存在，不变）
default = torch.cuda.default_stream()

# current_stream(): 获取当前线程在该设备上的"活跃流"（可改变）
current = torch.cuda.current_stream()

print(default == current)  # True（初始时）

# 切换当前流
s = torch.cuda.Stream()
torch.cuda.set_stream(s)

current_now = torch.cuda.current_stream()
print(default == current_now)      # False
print(current_now == s)            # True

# 但 default_stream() 始终返回同一个对象
print(torch.cuda.default_stream() == default)  # True

## 总结
torch.cuda.default_stream() 的核心要点：
- 每个 CUDA 设备有一个默认流，它是所有未显式指定流的 CUDA 操作的"归宿"
- 默认流是隐式的：tensor.cuda()、model.to('cuda')、普通张量运算都在其上执行
- 同步控制的关键：通过 wait_stream() 可以让其他流等待默认流，或反之
- 性能优化的基准：多流并发优化的参照点，理解默认流行为是掌握 CUDA 异步执行的基础